In [22]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split

import statsmodels.formula.api as smf
import statsmodels.api as sm

from sklearn.metrics import mean_poisson_deviance

In [ ]:
# Read the cleaned data
root = Path.cwd().parent
data_path = root / "data/cleaned_data.csv"

df = pd.read_csv(data_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 679513 entries, 0 to 679512
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id               679513 non-null  float64
 1   n_claims         679513 non-null  float64
 2   exposure         679513 non-null  float64
 3   area             679513 non-null  object 
 4   vehicle_power    679513 non-null  float64
 5   vehicle_age      679513 non-null  float64
 6   driver_age       679513 non-null  float64
 7   bonus_malus      679513 non-null  float64
 8   vehicle_brand    679513 non-null  object 
 9   vehicle_gas      679513 non-null  object 
 10  density          679513 non-null  float64
 11  region           679513 non-null  object 
 12  claim_amount     679513 non-null  float64
 13  claim_frequency  679513 non-null  float64
 14  claim_severity   679513 non-null  float64
 15  pure_premium     679513 non-null  float64
dtypes: float64(12), object(4)
memory usage

In [ ]:
# Take logarithm of density because little changes in low density
# affects more than little changes in high density. 
df["log_density"] = np.log(df["density"])

# Convert object columns to categories.
categorical_cols = ["area", "vehicle_brand", "vehicle_gas", "region"]
for col in categorical_cols:
    df[col] = df[col].astype("category")

In [ ]:
# Split the data into train and test data.
train_df, test_df = train_test_split(df, test_size=0.2, random_state=13) 

## POISSON GLM

In [20]:
poisson_formula = """ 
n_claims ~  area +
            vehicle_power + 
            vehicle_age + 
            driver_age + 
            bonus_malus +
            vehicle_brand +
            vehicle_gas +
            log_density
"""

poisson_model = smf.glm(
    formula=poisson_formula, 
    data=train_df,
    family=sm.families.Poisson(),
    offset=np.log(train_df["exposure"])
).fit()

In [21]:
poisson_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:               n_claims   No. Observations:               543610
Model:                            GLM   Df Residuals:                   543588
Model Family:                 Poisson   Df Model:                           21
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.2500e+05
Date:                Tue, 23 Dec 2025   Deviance:                   1.9102e+05
Time:                        21:36:31   Pearson chi2:                 1.90e+06
No. Iterations:                     7   Pseudo R-squ. (CS):            0.01208
Covariance Type:            nonrobust                                         
==========================================================================================
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -3.9248      0.060    -65.600      0.000      -4.042      -3.808
area[T.B]                  0.0277      0.027      1.014      0.311      -0.026       0.081
area[T.C]                  0.0617      0.035      1.759      0.079      -0.007       0.130
area[T.D]                  0.1956      0.053      3.686      0.000       0.092       0.300
area[T.E]                  0.1738      0.071      2.458      0.014       0.035       0.312
area[T.F]                  0.1387      0.097      1.435      0.151      -0.051       0.328
vehicle_brand[T.B10]      -0.0753      0.040     -1.894      0.058      -0.153       0.003
vehicle_brand[T.B11]       0.0498      0.042      1.188      0.235      -0.032       0.132
vehicle_brand[T.B12]       0.0906      0.018      5.136      0.000       0.056       0.125
vehicle_brand[T.B13]      -0.0082      0.043     -0.190      0.849      -0.093       0.077
vehicle_brand[T.B14]      -0.2488      0.085     -2.936      0.003      -0.415      -0.083
vehicle_brand[T.B2]       -0.0766      0.016     -4.745      0.000      -0.108      -0.045
vehicle_brand[T.B3]       -0.0592      0.023     -2.557      0.011      -0.105      -0.014
vehicle_brand[T.B4]       -0.1093      0.032     -3.437      0.001      -0.172      -0.047
vehicle_brand[T.B5]        0.0045      0.026      0.170      0.865      -0.047       0.056
vehicle_brand[T.B6]       -0.1040      0.030     -3.442      0.001      -0.163      -0.045
vehicle_gas[T.Regular]     0.0372      0.012      3.204      0.001       0.014       0.060
vehicle_power              0.0066      0.003      2.234      0.025       0.001       0.012
vehicle_age               -0.0339      0.001    -27.635      0.000      -0.036      -0.032
driver_age                 0.0072      0.000     17.270      0.000       0.006       0.008
bonus_malus                0.0231      0.000     72.917      0.000       0.023       0.024
log_density                0.0138      0.013      1.038      0.299      -0.012       0.040
==========================================================================================
"""

In [23]:
test_df["poisson_pred"] = poisson_model.predict(test_df, offset=np.log(test_df["exposure"]))

poisson_dev = mean_poisson_deviance(test_df["n_claims"], test_df["poisson_pred"])

poisson_dev

0.3521292274958408

In [25]:
obs = test_df["n_claims"].sum()
pred = test_df["poisson_pred"].sum()

relative_err = abs(obs-pred)/obs
relative_err

np.float64(0.011972384928047329)

## NEGATIVE BINOMIAL GLM

In [30]:
nb_model = smf.glm(
    formula=poisson_formula,
    data=train_df,
    family=sm.families.NegativeBinomial(),
    offset=np.log(train_df["exposure"])                   
).fit()

/home/necati/Documents/Auto-Insurance-Modelling/venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [31]:
nb_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:               n_claims   No. Observations:               543610
Model:                            GLM   Df Residuals:                   543588
Model Family:        NegativeBinomial   Df Model:                           21
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.2320e+05
Date:                Tue, 23 Dec 2025   Deviance:                   1.6403e+05
Time:                        22:16:43   Pearson chi2:                 1.83e+06
No. Iterations:                     8   Pseudo R-squ. (CS):            0.01119
Covariance Type:            nonrobust                                         
==========================================================================================
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -3.9164      0.063    -62.150      0.000      -4.040      -3.793
area[T.B]                  0.0300      0.028      1.058      0.290      -0.026       0.086
area[T.C]                  0.0667      0.036      1.831      0.067      -0.005       0.138
area[T.D]                  0.2116      0.055      3.836      0.000       0.103       0.320
area[T.E]                  0.1848      0.074      2.513      0.012       0.041       0.329
area[T.F]                  0.1600      0.101      1.591      0.112      -0.037       0.357
vehicle_brand[T.B10]      -0.0746      0.041     -1.805      0.071      -0.156       0.006
vehicle_brand[T.B11]       0.0457      0.044      1.044      0.296      -0.040       0.132
vehicle_brand[T.B12]       0.1075      0.018      5.859      0.000       0.072       0.143
vehicle_brand[T.B13]      -0.0137      0.045     -0.302      0.763      -0.103       0.075
vehicle_brand[T.B14]      -0.2589      0.088     -2.953      0.003      -0.431      -0.087
vehicle_brand[T.B2]       -0.0812      0.017     -4.829      0.000      -0.114      -0.048
vehicle_brand[T.B3]       -0.0636      0.024     -2.635      0.008      -0.111      -0.016
vehicle_brand[T.B4]       -0.1141      0.033     -3.446      0.001      -0.179      -0.049
vehicle_brand[T.B5]        0.0026      0.027      0.095      0.925      -0.051       0.056
vehicle_brand[T.B6]       -0.1105      0.031     -3.509      0.000      -0.172      -0.049
vehicle_gas[T.Regular]     0.0456      0.012      3.766      0.000       0.022       0.069
vehicle_power              0.0046      0.003      1.492      0.136      -0.001       0.011
vehicle_age               -0.0345      0.001    -27.019      0.000      -0.037      -0.032
driver_age                 0.0072      0.000     16.499      0.000       0.006       0.008
bonus_malus                0.0234      0.000     66.765      0.000       0.023       0.024
log_density                0.0117      0.014      0.849      0.396      -0.015       0.039
==========================================================================================
"""

In [34]:
test_df["nb_pred"] = nb_model.predict(test_df, offset=np.log(test_df["exposure"]))

nb_dev = mean_poisson_deviance(test_df["n_claims"], test_df["nb_pred"])

nb_dev

0.3521529081043275

In [35]:
# Since nb_model did not reduce deviance, keep poisson_model
test_df[["id", "poisson_pred"]].to_csv(root / "data/frequency_predictions", index=False)